# Encontro 6: Avaliação de Chatbots — Métricas, Dataset e Ferramentas

Neste encontro, aprendemos na prática como avaliar um chatbot/RAG:
- Definir métricas: **Correctness**, **Faithfulness**, **Answer Relevancy**.
- Criar um **dataset de avaliação** (perguntas e respostas esperadas).
- Usar **frameworks** como RAGAS e LangSmith Evals.


In [ ]:
%pip install -q -r ../requirements.txt
%pip install -q ragas datasets evaluate rouge-score langsmith


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME", "gemini-2.0-flash")
assert API_KEY, "GOOGLE_API_KEY ausente. Defina no .env."

# LLM para gerar respostas no pipeline avaliado
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, google_api_key=API_KEY, streaming=False)
# Embeddings para indexação e métricas baseadas em similaridade
embeddings = GoogleGenerativeAIEmbeddings(google_api_key=API_KEY, model="models/text-embedding-004")
print("Ambiente pronto.")


## 1) Métricas de Avaliação

- **Correctness** (correção): A resposta está correta em relação ao gabarito esperado?
  - Ex.: comparar com respostas esperadas via ROUGE/semântica.
- **Faithfulness** (fidelidade): A resposta é **suportada** pelo contexto recuperado (não inventa fatos)?
  - Ex.: avaliar grounding com LLM ou checagens de similaridade com o contexto.
- **Answer Relevancy** (relevância): A resposta endereça de fato a pergunta feita?
  - Ex.: avaliar relação pergunta ↔ resposta por embeddings/LLM.


## 2) Dataset de Avaliação

Vamos criar um pequeno dataset com perguntas e respostas esperadas (gabarito) usando a base `ingestion/base.md`.
Também geraremos **contexts** com chunks para simular o que o RAG recupera.


In [ ]:
from langchain_core.documents import Document

# Carrega e split da base de conhecimento
with open("../ingestion/base.md", "r", encoding="utf-8") as f:
    base_text = f.read()
docs = [Document(page_content=base_text, metadata={"source": "base.md"})]
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Chunks: {len(chunks)}")

# Dataset com perguntas e gabaritos
eval_dataset = [
    {
        "question": "Quais passos devo validar antes de um deploy?",
        "ground_truth": "Executar testes, validar variáveis/secrets, configurar monitoramento, planejar rollback e comunicar manutenção.",
    },
    {
        "question": "Quando um ticket deve ter prioridade alta?",
        "ground_truth": "Para incidentes que afetam produção (impacto crítico em usuários/sistema).",
    },
    {
        "question": "Como tratar erros 401 em uma API?",
        "ground_truth": "Verificar autenticação/chave de API, permissões e limites de taxa; implementar retries se necessário.",
    },
]
print(f"Itens de avaliação: {len(eval_dataset)}")


## 3) Pipeline sob avaliação (RAG)

Indexamos os chunks com Chroma e criamos um `retriever`. Depois, montamos um chain simples com `PromptTemplate`.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

persist_dir = "./chroma_eval"
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=persist_dir)
vectorstore.persist()
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

prompt = PromptTemplate.from_template(
    "Você é um assistente de suporte. Responda usando estritamente o contexto.\n\nContexto:\n{context}\n\nPergunta: {question}"
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | RunnableLambda(lambda m: getattr(m, "content", m))
    | StrOutputParser()
)
print("RAG pronto para avaliação.")


## 4) Gerar respostas do pipeline

Para cada item do dataset, geramos a resposta e também guardamos os **contexts** usados.


In [ ]:
predictions = []
contexts_per_item = []
for item in eval_dataset:
    q = item["question"]
    # obter contexts (docs) para registro
    docs_ctx = retriever.invoke(q)
    contexts_per_item.append([d.page_content for d in docs_ctx])
    # resposta do chain
    ans = rag_chain.invoke(q)
    predictions.append(ans)

print("Exemplo de resposta:", predictions[0][:120], "...")
print("Contexts (amostra):", contexts_per_item[0][0][:80].replace("\n", " "), "...")


## 5) Métricas locais (sem frameworks)

Implementamos versões simples das métricas para entendimento:
- **Correctness (ROUGE-L)**: sobreposição lexical com gabarito.
- **Answer Relevancy (cosine)**: similaridade entre resposta e pergunta por embeddings.
- **Faithfulness (support score)**: média de similaridade da resposta com contexts.

Obs.: São aproximações — frameworks como RAGAS fazem avaliação de grounding mais robusta com LLMs.


In [ ]:
import numpy as np
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def cosine(a, b):
    a = np.array(a); b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

results = []
for i, item in enumerate(eval_dataset):
    gt = item["ground_truth"]
    pred = predictions[i]
    q = item["question"]
    ctxs = contexts_per_item[i]

    # Correctness por ROUGE-L (rouge-score)
    scores = scorer.score(gt, pred)
    correctness = scores["rougeL"].fmeasure

    # Answer Relevancy por embeddings (pergunta vs resposta)
    emb_q = embeddings.embed_query(q)
    emb_a = embeddings.embed_query(pred)
    relevancy = cosine(emb_q, emb_a)

    # Faithfulness: média da similaridade da resposta com cada contexto
    sims = []
    emb_ans = emb_a
    for c in ctxs:
        emb_c = embeddings.embed_query(c)
        sims.append(cosine(emb_ans, emb_c))
    faithfulness = float(np.mean(sims)) if sims else 0.0

    results.append({
        "correctness_rougeL": correctness,
        "answer_relevancy_cosine": relevancy,
        "faithfulness_mean_cosine": faithfulness,
    })

for i, r in enumerate(results, 1):
    print(f"Item {i}: " + str(r))


## 5.1) Resumo das Métricas
Médias agregadas de **Correctness (ROUGE-L)**, **Answer Relevancy** e **Faithfulness** e export para CSV.


In [ ]:
import numpy as np
import os, csv

if not results:
    print("Nenhum resultado disponível. Execute a geração de predictions primeiro.")
else:
    avg_correctness = float(np.mean([r["correctness_rougeL"] for r in results]))
    avg_relevancy = float(np.mean([r["answer_relevancy_cosine"] for r in results]))
    avg_faithfulness = float(np.mean([r["faithfulness_mean_cosine"] for r in results]))

    print("Médias (0–1):")
    print("- Correctness (ROUGE-L f):", round(avg_correctness, 4))
    print("- Answer Relevancy (cosine):", round(avg_relevancy, 4))
    print("- Faithfulness (mean cosine):", round(avg_faithfulness, 4))

    # Exporta CSV com detalhes por item
    out_path = os.path.join("notebooks", "chroma_eval", "eval_results_encontro6.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "question", "ground_truth", "prediction",
            "correctness_rougeL", "answer_relevancy_cosine", "faithfulness_mean_cosine"
        ])
        writer.writeheader()
        for i, item in enumerate(eval_dataset):
            writer.writerow({
                "question": item.get("question", ""),
                "ground_truth": item.get("ground_truth", ""),
                "prediction": predictions[i],
                "correctness_rougeL": results[i]["correctness_rougeL"],
                "answer_relevancy_cosine": results[i]["answer_relevancy_cosine"],
                "faithfulness_mean_cosine": results[i]["faithfulness_mean_cosine"]
            })
    print("Relatório CSV salvo em:", out_path)


## 6) Framework RAGAS (opcional)

RAGAS fornece métricas como **faithfulness** e **answer relevancy** baseadas em LLMs e heurísticas.
Você precisa configurar uma chave de provedor compatível para execução.


In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy
    print("RAGAS importado.")
except Exception as e:
    print("RAGAS não disponível:", e)

import os

# Prepara dataset para RAGAS
ragas_data = []
for i, item in enumerate(eval_dataset):
    ragas_data.append({
        "question": item["question"],
        "answer": predictions[i],
        "contexts": contexts_per_item[i],
        "ground_truth": item["ground_truth"],
    })

# Converte para HuggingFace Dataset (formato esperado pelo RAGAS)
from datasets import Dataset as HFDataset
ragas_dataset = HFDataset.from_list(ragas_data)

# Executa avaliação (RAGAS usa OpenAI por padrão)
# Executa avaliação com Gemini via LangChain (wrappers se disponíveis)
google_key = os.environ.get("GOOGLE_API_KEY")
if not google_key:
    print("RAGAS: GOOGLE_API_KEY não definido. Configure para usar Gemini.")
    print("Ex.: os.environ['GOOGLE_API_KEY']='SUA_CHAVE'")
else:
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
        chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
        emb_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
        # Tenta usar os wrappers do RAGAS; se não houver, passa instâncias diretamente
        try:
            from ragas.llms import LangchainLLMWrapper
            from ragas.embeddings import LangchainEmbeddingsWrapper
            llm = LangchainLLMWrapper(chat)
            emb = LangchainEmbeddingsWrapper(emb_model)
        except Exception as wrap_err:
            print("Wrappers LangChain do RAGAS indisponíveis:", wrap_err)
            print("Usando instâncias do LangChain diretamente.")
            llm = chat
            emb = emb_model
        ragas_report = evaluate(ragas_dataset, metrics=[faithfulness, answer_relevancy], llm=llm, embeddings=emb)
        print(ragas_report)
    except Exception as e:
        print("Falha ao rodar RAGAS com Gemini (via LangChain):", e)


## 7) LangSmith Evals (opcional)

LangSmith permite criar datasets e rodar avaliações com rastreamento de runs.
Para usar:
- Configure suas chaves de acesso e projeto.
- Defina um runner/functon para seu pipeline.
- Execute os evaluators (correctness/consistency/relevancy).


In [ ]:
try:
    from langsmith import Client
    print("LangSmith importado.")
except Exception as e:
    print("LangSmith não disponível:", e)

# Exemplo mínimo (skeleton): criar dataset e registrar previsões do pipeline
client = Client()
# API espera 'dataset_name' (ou positional) em versões atuais
ds = client.create_dataset(dataset_name="eval-chatbot-suporte")
print("Dataset criado:", ds.id)
for item in eval_dataset:
    client.create_example(inputs={"question": item["question"]}, outputs={"ground_truth": item["ground_truth"]}, dataset_id=ds.id)

# # Depois, rodar seu runner e associar resultados ao dataset
# # Veja documentação oficial para configurar evaluators e reports.

print("Skeleton de LangSmith pronto (configure chaves e projeto para usar).")


## 8) Conclusões

- Métricas ajudam a acompanhar evolução e qualidade do chatbot.
- **Correctness** avalia alinhamento com gabarito; **Faithfulness** evita alucinações; **Relevancy** verifica aderência à pergunta.
- Frameworks como **RAGAS** e **LangSmith** aceleram e padronizam a avaliação.
- Mantenha um *dataset* atualizado e avalie frequentemente.
